In [ ]:
import kagglehub
#Lets Start with the imports
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
import os
import numpy as np
import kagglehub
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')




# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
cvs = os.path.join(path , 'Q3_data.csv')
df = pd.read_csv(cvs)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# TWO handels one for the others and one for the target
# for the other features  Even though doesnt have it i'll add it add and not remove
print("\nMissing Values (df.isnull().sum()):")
print(df.isnull().sum())
print(df.isnull().sum().sum())
# Ill fill them with the The most coomen for all the object coulms
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
  df[col]=df[col].fillna(df[col].mode()[0])

# for the float I'll get the meanian
num = df.select_dtypes(include=["float64", "int64"]).columns
df[num]= df[num].fillna(df[num].median())
# Target has to be removed
df.dropna(subset=['Target'])


In [ ]:
print("\nMissing Values (df.isnull().sum()):")
print(df.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols)) # no encodined needed

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 5: Write your code here:
# Delivery_Time  distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Target'].dropna(), bins=50, edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.show() # very Skewed

In [ ]:
from sklearn.model_selection import KFold , train_test_split , StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
# Task 1: Write your code here:
X = df.drop('Target', axis = 1)
y= df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify= y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Legendary in train: {y_train.sum()}, in test: {y_test.sum()}")

In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost  -q

clear_output()

In [ ]:
# Task 2,3,4,5: Write your code here:

from catboost import CatBoostClassifier
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
n_splits = 5 # K
f11= []
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  f1 = f1_score(y_test, y_pred, zero_division=0)
  f11.append(f1)
print(f11)

In [ ]:
f11

In [ ]:
# Task 1: Write your code here:
importances = {}

#importances['Random Forest'] = sklearn_models['Random Forest'].feature_importances_
#importances['XGBoost'] = sklearn_models['XGBoost'].feature_importances_
importances['CatBoost'] = model.feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(3,1, figsize=(20, 100))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

#plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
print('p_2 is the most importat feature ')

In [ ]:
# Task Bonus: Write your code here:
from sklearn.model_selection import KFold , train_test_split , StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
# Task 1: Write your code here:
X = df['P_2']
y= df['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify= y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Legendary in train: {y_train.sum()}, in test: {y_test.sum()}")

# Task 2,3,4,5: Write your code here:

from catboost import CatBoostClassifier
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
n_splits = 5 # K
f11= []
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
  f1 = f1_score(y_test, y_pred, zero_division=0)
  f11.append(f1)
print(f11)